In [1]:
# pip install pandas scikit-learn joblib

import pandas as pd
from joblib import dump
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

# ---------------------------
# 1) Load your data
# ---------------------------
# Put your dataset path here
DATA_PATH = "data.csv"

# Set your target column name here
TARGET_COL = "target"

df = pd.read_csv(DATA_PATH)



FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

In [ ]:
# ---------------------------
# 2) Split features/target
# ---------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

In [ ]:
# ---------------------------
# 3) Build preprocessing
# ---------------------------
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", categorical_preprocess, cat_cols),
    ],
    remainder="drop"
)



In [ ]:
# ---------------------------
# 4) Model
# ---------------------------
model = LogisticRegression(max_iter=2000)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])

In [ ]:
# ---------------------------
# 5) Train
# ---------------------------
clf.fit(X_train, y_train)

In [ ]:
# ---------------------------
# 6) Evaluate
# ---------------------------
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))



In [ ]:
# ---------------------------
# 7) Save pipeline (preprocess + model)
# ---------------------------
dump(clf, "basic_model_pipeline.joblib")
print("\nSaved: basic_model_pipeline.joblib")